# BiasAuditFW 2.0 — notebook oficial para Google Colab T4

Este notebook executa uma análise exploratória de diversidade percebida em imagens
geradas por IA. Há duas unidades de análise:

- `audit_images.csv`: uma linha por imagem, com os escores CLIP.
- `audit_faces.csv`: uma linha por rosto detectado, sem replicar escores CLIP.

O CLIP mede alinhamento semântico com protótipos de diversidade racial percebida e
diversidade de apresentação de gênero. Os escores não representam identidade,
ancestralidade, sexo biológico ou proporções demográficas.

## 1. Preparação

Antes de executar, selecione **Ambiente de execução > Alterar tipo de ambiente de
execução > T4 GPU**. O notebook instala o pacote diretamente do repositório para
evitar cópias divergentes das funções.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/Kauandugi/bias-audit-framework.git"
REPOSITORY_REF = "agent/schema-2-diversity-refactor"
REPO_DIR = Path("/content/bias-audit-framework")

if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
    raise RuntimeError(
        f"{REPO_DIR} existe, mas não é um clone Git. "
        "Remova ou renomeie esse diretório e execute a célula novamente."
    )

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--quiet", "--no-checkout", REPOSITORY_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(REPO_DIR), "fetch", "--quiet", "--depth", "1", "origin", REPOSITORY_REF],
    check=True,
)
subprocess.run(
    ["git", "-C", str(REPO_DIR), "checkout", "--quiet", "--force", "FETCH_HEAD"],
    check=True,
)
REPOSITORY_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError(
        f"A revisão {REPOSITORY_COMMIT} não contém pyproject.toml. "
        f"Verifique REPOSITORY_REF={REPOSITORY_REF!r}."
    )
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--editable",
        f"{REPO_DIR}[neural,dashboard,test]",
    ],
    check=True,
)

import biasauditfw
from biasauditfw import SCHEMA_VERSION

print(f"Revisão do repositório: {REPOSITORY_COMMIT}")
print(f"BiasAuditFW schema {SCHEMA_VERSION} carregado de {biasauditfw.__file__}")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Configuração da execução

O perfil original usa o manifesto versionado com 64 imagens. Para outro dataset,
defina `MANIFEST_PATH = None` e `EXPECTED_IMAGES = None`. Sem metadados explícitos,
o pipeline gera estatística descritiva e informa por que não executou inferência.

In [ ]:
from pathlib import Path

DATASET_ROOT = Path("/content/drive/MyDrive/Estudo UNAL/Images Generated by IA")
OUTPUT_ROOT = Path("/content/drive/MyDrive/Estudo UNAL/BiasAuditFW outputs schema 2")
DATASET_ID = "original-64"
MANIFEST_PATH = REPO_DIR / "data" / "original_64_manifest.csv"
EXPECTED_IMAGES = 64

RUN_FULL = False
RUN_GROUND_TRUTH = False
RUN_HUMAN_BASELINE = False
RUN_FAIRFACE = False
DOWNLOAD_FAIRFACE_WEIGHTS = False

GROUND_TRUTH_COCO = Path("/content/drive/MyDrive/Estudo UNAL/ground_truth_consenso.json")
FAIRFACE_WEIGHTS = Path("/content/fairface_alldata_20191111.pt")
HUMAN_BASELINE_URL = (
    "https://raw.githubusercontent.com/Kauandugi/"
    "Bias-ai-university-visuals/main/data/metadada/"
    "Evaluation%20Phase%20-%20Paper%20-%20Evaluation.csv"
)

In [ ]:
import random

import numpy as np
import pandas as pd
import torch

from biasauditfw import DatasetConfig, discover_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CONFIG = DatasetConfig(
    input_root=DATASET_ROOT,
    output_dir=OUTPUT_ROOT,
    dataset_id=DATASET_ID,
    manifest_path=MANIFEST_PATH,
    expected_images=EXPECTED_IMAGES,
    seed=SEED,
)
RUN_OUTPUT = OUTPUT_ROOT / ("full" if RUN_FULL else "smoke")
RUN_OUTPUT.mkdir(parents=True, exist_ok=True)

## 3. Preflight de ingestão

A busca é recursiva e não interpreta posições de pastas como modelo ou prompt.
Imagens corrompidas, duplicadas, links simbólicos e referências inválidas no
manifesto interrompem a execução antes do carregamento dos modelos.

In [ ]:
discovery = discover_dataset(CONFIG)
inventory = discovery.images

print(f"Imagens únicas válidas: {len(inventory)}")
print(f"Diretório de saída desta execução: {RUN_OUTPUT}")
display(inventory.drop(columns=["absolute_path"]).head())
display(discovery.report["status_ingestao"].value_counts(dropna=False))

## 4. Testes determinísticos

Estes testes não baixam pesos neurais. Eles verificam ingestão, schema 1:N,
normalização vetorial, estatística, IoU, Ground Truth e pseudo-oráculo simulado.

In [ ]:
!python -m pytest -q {REPO_DIR / "tests"} -m "not gpu" --cov=biasauditfw --cov-report=term

## 5. Carregamento único dos modelos

Os protótipos textuais do CLIP são codificados uma única vez. Cada frase é
normalizada por L2; os vetores de cada família são promediados e normalizados
novamente. A imagem também é normalizada antes do produto escalar.

In [ ]:
from functools import partial

from biasauditfw.clip_diversity import ClipDiversityScorer
from biasauditfw.faces import analyze_faces

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(f"Dispositivo: {DEVICE}")
print(f"GPU: {GPU_NAME}")
if DEVICE.type != "cuda":
    print("ATENÇÃO: selecione uma GPU T4 antes da execução neural completa.")

clip_scorer = ClipDiversityScorer(CONFIG.clip_model_id, DEVICE)
face_analyzer = partial(analyze_faces, detector_backend=CONFIG.detector_backend)
print("CLIP carregado; protótipos textuais armazenados em cache.")

## 6. Smoke test ou corpus completo

Com `RUN_FULL = False`, apenas quatro imagens são processadas. Depois de conferir
os artefatos, altere para `True` e execute novamente a partir da configuração.

In [ ]:
import importlib.metadata as package_metadata
import platform

from biasauditfw import export_results, process_dataset, validate_contract

limit = None if RUN_FULL else CONFIG.smoke_images
audit_images, audit_faces = process_dataset(
    inventory,
    face_analyzer=face_analyzer,
    semantic_scorer=clip_scorer,
    max_images=limit,
)
expected_for_run = EXPECTED_IMAGES if RUN_FULL else min(CONFIG.smoke_images, len(inventory))
validate_contract(audit_images, audit_faces, expected_for_run)

RUN_METADATA = {
    "dataset_id": DATASET_ID,
    "run_mode": "full" if RUN_FULL else "smoke",
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": package_metadata.version("transformers"),
    "deepface_version": package_metadata.version("deepface"),
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": GPU_NAME,
    "clip_model_id": CONFIG.clip_model_id,
    "detector_backend": CONFIG.detector_backend,
    "seed": SEED,
    "clip_scoring": "L2-normalized cosine prototype margins",
    "manifest_path": str(MANIFEST_PATH) if MANIFEST_PATH else None,
}
export_results(
    audit_images,
    audit_faces,
    discovery.report,
    RUN_OUTPUT,
    RUN_METADATA,
    expected_images=expected_for_run,
)
print(f"Imagens: {len(audit_images)} | rostos: {len(audit_faces)}")
display(audit_images.head())
display(audit_faces.head())

## 7. Estatística no nível da imagem

Mann–Whitney e Wilcoxon são executados separadamente para as duas margens. A
correção de Holm cobre as duas hipóteses primárias. Sem manifesto elegível, somente
resultados descritivos são exportados.

In [ ]:
from biasauditfw.reporting import export_statistical_reports

inference_status = export_statistical_reports(
    audit_images,
    audit_faces,
    RUN_OUTPUT,
)
print(inference_status)
if inference_status["eligible"]:
    display(pd.read_csv(RUN_OUTPUT / "mann_whitney_primary.csv"))
    display(pd.read_csv(RUN_OUTPUT / "wilcoxon_sensitivity.csv"))

## 8. Gráficos exploratórios

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
long_clip = audit_images.melt(
    id_vars=["imagem_id", "tipo_prompt", "modelo_ia"],
    value_vars=[
        "clip_racial_diversity_margin",
        "clip_gender_diversity_margin",
    ],
    var_name="dimensao",
    value_name="margem",
)
long_clip["dimensao"] = long_clip["dimensao"].map(
    {
        "clip_racial_diversity_margin": "Diversidade racial percebida",
        "clip_gender_diversity_margin": "Apresentação de gênero",
    }
)
plt.figure(figsize=(10, 6))
sns.boxplot(data=long_clip, x="dimensao", y="margem", hue="tipo_prompt")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("")
plt.ylabel("Margem CLIP (diversidade - homogeneidade)")
plt.tight_layout()
plt.savefig(RUN_OUTPUT / "grafico_margens_clip.png", dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=audit_images, x="modelo_ia", y="num_faces")
plt.xlabel("Modelo gerador")
plt.ylabel("Rostos detectados por imagem")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(RUN_OUTPUT / "grafico_num_faces.png", dpi=300)
plt.show()

## 9. Ground Truth da detecção facial

Ative somente após consolidar as caixas dos dois avaliadores em COCO. O
pareamento usa caminho relativo ou `source_image_id`, com correspondência
um-para-um e `IoU >= 0,5`.

In [ ]:
from biasauditfw.validation import (
    bootstrap_detector,
    coco_to_dataframe,
    evaluate_detector,
)

if RUN_GROUND_TRUTH:
    ground_truth_faces = coco_to_dataframe(
        GROUND_TRUTH_COCO, audit_images, "consenso"
    )
    detector_per_image, detector_metrics = evaluate_detector(
        audit_faces, ground_truth_faces, audit_images
    )
    detector_ci = bootstrap_detector(detector_per_image, seed=SEED)
    ground_truth_faces.to_csv(RUN_OUTPUT / "ground_truth_faces.csv", index=False)
    detector_per_image.to_csv(RUN_OUTPUT / "detector_per_image.csv", index=False)
    detector_metrics.to_csv(RUN_OUTPUT / "detector_metrics.csv", index=False)
    detector_ci.to_csv(RUN_OUTPUT / "detector_bootstrap_ci.csv", index=False)
    display(detector_metrics)
    display(detector_ci)
else:
    print("Ground Truth desativado: aguardando anotações humanas consolidadas.")

## 10. Baseline humana

As avaliações do artigo anterior são usadas como evidência convergente:
`Racial_Diversity` para a margem racial e `Gender_Representation` para a margem
de apresentação de gênero.

In [ ]:
from biasauditfw.statistics import (
    human_clip_correlations,
    human_interrater_report,
)

if RUN_HUMAN_BASELINE:
    human = pd.read_csv(HUMAN_BASELINE_URL)
    human = human.loc[human["Image_id"].notna()].copy()
    interrater = human_interrater_report(human)
    human_clip = human_clip_correlations(human, audit_images)
    human.to_csv(RUN_OUTPUT / "human_baseline.csv", index=False)
    interrater.to_csv(RUN_OUTPUT / "human_interrater.csv", index=False)
    human_clip.to_csv(RUN_OUTPUT / "human_clip_correlations.csv", index=False)
    display(interrater)
    display(human_clip)
else:
    print("Baseline humana desativada.")

## 11. FairFace como pseudo-oráculo

FairFace é uma referência secundária e também pode reproduzir vieses. Ele é
executado nos recortes do Ground Truth, não nas caixas produzidas pelo RetinaFace.

In [ ]:
from biasauditfw.neural import (
    FairFaceOracle,
    align_deepface_to_ground_truth,
    download_fairface_weights,
    run_fairface_on_ground_truth,
)
from biasauditfw.validation import concordance_report

if DOWNLOAD_FAIRFACE_WEIGHTS:
    download_fairface_weights(FAIRFACE_WEIGHTS)

if RUN_FAIRFACE:
    if not RUN_GROUND_TRUTH:
        raise RuntimeError("RUN_FAIRFACE exige RUN_GROUND_TRUTH = True.")
    fairface_oracle = FairFaceOracle(FAIRFACE_WEIGHTS, DEVICE)
    fairface_predictions = run_fairface_on_ground_truth(
        fairface_oracle,
        ground_truth_faces,
        audit_images,
        CONFIG.input_root,
    )
    aligned = align_deepface_to_ground_truth(audit_faces, ground_truth_faces)
    comparison = aligned.merge(
        fairface_predictions,
        on=["imagem_id", "gt_face_id"],
        validate="one_to_one",
    )
    concordance, matrices = concordance_report(comparison)
    fairface_predictions.to_csv(
        RUN_OUTPUT / "fairface_predictions.csv", index=False
    )
    comparison.to_csv(RUN_OUTPUT / "deepface_fairface_aligned.csv", index=False)
    concordance.to_csv(
        RUN_OUTPUT / "deepface_fairface_concordance.csv", index=False
    )
    for name, matrix in matrices.items():
        matrix.to_csv(RUN_OUTPUT / f"confusion_{name}.csv")
    display(concordance)
else:
    print("FairFace desativado.")

## 12. Validação independente dos outputs

Na execução completa do corpus original, use `--expected-images 64
--require-cuda`. O validador recalcula os testes estatísticos a partir do CSV.

In [ ]:
validator_args = f'"{RUN_OUTPUT}" --expected-images {expected_for_run}'
if RUN_FULL:
    validator_args += " --require-cuda"
!python {REPO_DIR / "tests" / "validate_full_outputs.py"} {validator_args}

## Checklist final

1. O smoke test deve gerar quatro imagens sem violar o schema.
2. A execução completa original deve gerar 64 `imagem_id` únicos.
3. A soma de `num_faces` deve coincidir com as linhas de `audit_faces.csv`.
4. Nenhuma coluna CLIP pode aparecer na tabela facial.
5. Os dois escores são alinhamentos semânticos, não medidas de identidade.
6. Ground Truth e FairFace continuam opcionais até as anotações estarem prontas.
7. Traga a pasta `full` de volta ao repositório antes de preencher resultados no TCC.